In [1]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import os
from os import listdir
from os.path import isfile, join
import re
import MasterPSD as PSD
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import glob
import allantools
%matplotlib qt
param_map = {
    '5ms': {'time': 5},
    '50ms': {'time': 0.5},
    'noms': {'time': 0.1},
    '5gs': {'time': 0.01},
    '250ks': {'time': 20}
}

In [60]:
OFF_I_02_LPFno_ts, OFF_I_02_LPFno_xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-19\PID_1_02_0_OFF_LPFno_19_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
OFF_I_02_LPFno_ts=OFF_I_02_LPFno_ts*(10**(-3))

In [64]:
ON_I_02_LPFno_ts, ON_I_02_LPFno_xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-19\PID_1_02_0_ON_LPFno_19_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ON_I_02_LPFno_ts=ON_I_02_LPFno_ts*(10**(-3))

In [10]:
OFF_I_02_LPFno_ts[1]-OFF_I_02_LPFno_ts[0]

1.013

In [12]:
OFF_I_02_LPFno_ts[2]-OFF_I_02_LPFno_ts[1]

0.9990000000000001

In [14]:
OFF_I_02_LPFno_ts[3]-OFF_I_02_LPFno_ts[2]

0.9980000000000002

In [3]:
import numpy as np
from dataclasses import dataclass

@dataclass
class SamplingReport:
    n_samples: int
    duration_s: float
    fs_nominal: float          # 1 / median Δt (robust)
    fs_regression: float       # slope from linear fit index→time
    dt_mean: float
    dt_median: float
    dt_std: float
    jitter_rms: float          # RMS of (Δt - median Δt)
    jitter_ppm_rms: float      # RMS jitter relative to median Δt [ppm]
    max_abs_jitter: float
    pct_within_tolerance: float  # % of intervals within tolerance_ppm
    monotonic_non_decreasing: bool
    duplicate_timestamps: int
    negative_steps: int
    estimated_missing_intervals: float
    uniform_enough: bool       # pass/fail against tolerance_ppm
    notes: str

def assess_uniform_sampling(timestamps, tolerance_ppm=1000.0):
    """
    Assess how uniformly a time array is sampled and estimate sample rate.
    
    Parameters
    ----------
    timestamps : array-like
        1D array of sample times [seconds], preferably float, any epoch.
    tolerance_ppm : float
        Allowed RMS jitter relative to the *median* Δt, in parts per million.
        Example: 1000 ppm = 0.1% RMS.
        
    Returns
    -------
    SamplingReport
        Dataclass with metrics and a uniform_enough boolean.
    """
    t = np.asarray(timestamps, dtype=float).ravel()
    if t.size < 3:
        raise ValueError("Need at least 3 timestamps to assess uniformity.")
    
    # Basic checks
    dt = np.diff(t)
    duplicate_timestamps = int(np.sum(dt == 0))
    negative_steps = int(np.sum(dt < 0))
    monotonic_non_decreasing = negative_steps == 0
    
    # Robust nominal step & sample rate
    dt_median = float(np.median(dt))
    dt_mean   = float(np.mean(dt))
    dt_std    = float(np.std(dt, ddof=1)) if dt.size > 1 else 0.0
    fs_nominal = 1.0 / dt_median if dt_median > 0 else np.nan
    
    # Linear regression of t ≈ a + b * n to estimate global slope (b ≈ Δt)
    n = np.arange(t.size, dtype=float)
    # Closed-form least squares on centered variables (stable)
    nc = n - n.mean()
    tc = t - t.mean()
    denom = np.dot(nc, nc)
    if denom == 0:
        raise ValueError("Degenerate input for regression.")
    slope = float(np.dot(nc, tc) / denom)   # seconds / sample
    fs_reg = 1.0 / slope if slope > 0 else np.nan
    
    # Jitter relative to the robust step (median)
    jitter = dt - dt_median
    jitter_rms = float(np.sqrt(np.mean(jitter**2)))
    jitter_ppm_rms = float(1e6 * (jitter_rms / dt_median)) if dt_median > 0 else np.inf
    max_abs_jitter = float(np.max(np.abs(jitter)))
    
    # Tolerance pass/fail and percentage within tolerance
    tol_abs = (tolerance_ppm * 1e-6) * dt_median
    within = np.abs(jitter) <= tol_abs
    pct_within = float(100.0 * np.mean(within))
    uniform_enough = (jitter_ppm_rms <= tolerance_ppm) and monotonic_non_decreasing
    
    # Duration and missing-intervals estimate
    duration = float(t[-1] - t[0])
    # Expected intervals if perfectly uniform at fs_nominal
    expected_intervals = duration * fs_nominal if np.isfinite(fs_nominal) else np.nan
    estimated_missing_intervals = float(expected_intervals - (t.size - 1)) if np.isfinite(expected_intervals) else np.nan
    
    # notes
    notes = []
    if duplicate_timestamps > 0:
        notes.append(f"{duplicate_timestamps} duplicate timestamp(s).")
    if negative_steps > 0:
        notes.append(f"{negative_steps} negative time step(s) detected (non-monotonic).")
    if not np.isfinite(fs_nominal) or fs_nominal <= 0:
        notes.append("non-positive median Δt; cannot determine fs_nominal.")
    if abs(fs_reg - fs_nominal) / fs_nominal > 0.01 and np.isfinite(fs_nominal) and np.isfinite(fs_reg):
        notes.append("Regression-based fs differs from nominal by >1% (possible drift).")
    if pct_within < 95:
        notes.append("Fewer than 95% of intervals meet the tolerance.")
    
    return SamplingReport(
        n_samples=int(t.size),
        duration_s=duration,
        fs_nominal=fs_nominal,
 fs_regression=fs_reg,
        dt_mean=dt_mean,
        dt_median=dt_median,
        dt_std=dt_std,
        jitter_rms=jitter_rms,
        jitter_ppm_rms=jitter_ppm_rms,
        max_abs_jitter=max_abs_jitter,
        pct_within_tolerance=pct_within,
        monotonic_non_decreasing=monotonic_non_decreasing,
        duplicate_timestamps=duplicate_timestamps,
        negative_steps=negative_steps,
        estimated_missing_intervals=estimated_missing_intervals,
        uniform_enough=uniform_enough,
        notes=" ".join(notes) if notes else "OK"
    )

In [34]:
# ----- New: resampling/snap-to-grid helper -----
@dataclass
class ResampleResult:
    t_uniform: np.ndarray      # uniform time grid [s]
    y_uniform: np.ndarray      # resampled values
    fs: float                  # resulting sample rate [Hz]
    dt: float                  # step [s]
    report: SamplingReport     # diagnostic report
    resampled: bool            # True if we actually resampled
    info: str                  # brief note about what happened

def _linear_interp(x, xp, fp):
    # numpy.interp with endpoints extrapolated as edge values
    y = np.interp(x, xp, fp)
    # Guard: if x outside xp range, np.interp clamps to edge values already
    return y

def _zoh_interp(x, xp, fp):
    # Zero-order hold (previous value). Assumes xp is strictly increasing.
    idx = np.searchsorted(xp, x, side='right') - 1
    idx = np.clip(idx, 0, len(fp) - 1)
    return fp[idx]

def resample_to_uniform(timestamps, values, tolerance_ppm=1000.0, method="linear",
                        snap_if_uniform=False, trim_end=True):
    """
    Snap (t, y) to a uniform grid IF sampling is not uniform enough.

    Parameters
    ----------
    timestamps : array-like (seconds, any epoch)
    values     : array-like (same length)
    tolerance_ppm : float
        RMS Δt tolerance relative to median Δt to deem sampling 'uniform'.
    method : {"linear","zoh"}
        Interpolation method for resampling (linear or zero-order hold).
    snap_if_uniform : bool
        If True, even when uniform_enough==True, will re-grid to exact uniform times.
        If False, returns original (t, y).
    trim_end : bool
        When building the grid, align to t[0] and stop before exceeding t[-1] by > dt/2.

    Returns
    -------
    ResampleResult
    """
    t = np.asarray(timestamps, dtype=float).ravel()
    y = np.asarray(values, dtype=float).ravel()
    if t.size != y.size:
        raise ValueError("timestamps and values must have the same length.")
    if t.size < 3:
        raise ValueError("Need at least 3 samples to resample.")

    # Sort by time; collapse exact duplicates by taking the last value
    order = np.argsort(t, kind="mergesort")
    t = t[order]
    y = y[order]
    unique_mask = np.concatenate(([True], np.diff(t) != 0))
    # If duplicates exist, keep the last occurrence
    if not unique_mask.all():
        # For consecutive duplicates, we want the last value per run
        _, last_idx = np.unique(t, return_index=False, return_inverse=False, return_counts=False), None
        # Simpler: take unique times and for each time, take the last y
        # Use searchsorted to find boundaries
        unique_t, idx_start = np.unique(t, return_index=True)
        idx_end = np.r_[idx_start[1:], t.size] - 1
        t = unique_t
        y = y[idx_end]

    report = assess_uniform_sampling(t, tolerance_ppm=tolerance_ppm)
    dt_nom = 1.0 / report.fs_nominal if np.isfinite(report.fs_nominal) and report.fs_nominal > 0 else None
    if dt_nom is None:
        raise ValueError("Cannot determine nominal dt/fs from timestamps.")

    need_resample = (not report.uniform_enough) or snap_if_uniform

    if not need_resample:
        return ResampleResult(
            t_uniform=t,
            y_uniform=y,
            fs=report.fs_nominal,
            dt=1.0/report.fs_nominal,
            report=report,
            resampled=False,
            info="Input sampling passed tolerance; returned original arrays."
        )

    # Build uniform grid: start at t0, step dt_nom, stop near tN
    t0, tN = t[0], t[-1]
    # number of steps: round to nearest integer count
    n_steps = int(np.floor((tN - t0)/dt_nom + (0.5 if trim_end else 0.0)))
    n_steps = max(n_steps, 1)
    t_uni = t0 + np.arange(n_steps + 1) * dt_nom

    # Choose interpolation
    if method.lower() == "linear":
        y_uni = _linear_interp(t_uni, t, y)
        interp_info = "linear"
    elif method.lower() in ("zoh", "zero", "hold"):
        y_uni = _zoh_interp(t_uni, t, y)
        interp_info = "zero-order hold"
    else:
        raise ValueError("Unknown method. Use 'linear' or 'zoh'.")

    return ResampleResult(
        t_uniform=t_uni,
        y_uniform=y_uni,
        fs=1.0/dt_nom,
        dt=dt_nom,
        report=report,
        resampled=True,
        info=f"Resampled to uniform grid using {interp_info} interpolation (dt≈{dt_nom:.6g}s)."
    )


In [72]:
result = resample_to_uniform(OFF_I_02_LPFno_ts,OFF_I_02_LPFno_xs, tolerance_ppm=1no)  # 0.15% RMS tolerance
print(result.info)
#print(f"Original N={len(t)} | Uniform N={len(result.t_uniform)} | fs={result.fs:.3f} Hz")

Resampled to uniform grid using linear interpolation (dt≈0.001s).


In [66]:
result2 = resample_to_uniform(ON_I_02_LPFno_ts,ON_I_02_LPFno_xs, tolerance_ppm=1no)  # 0.15% RMS tolerance


In [70]:
print(result2.info)


Resampled to uniform grid using linear interpolation (dt≈0.001s).


In [75]:
print(result)

ResampleResult(t_uniform=array([1.6500000e-03, 2.6500000e-03, 3.6500000e-03, ..., 4.9999765e+02,
       4.9999865e+02, 4.9999965e+02]), y_uniform=array([3.01330000e-05, 3.01330000e-05, 3.01300360e-05, ...,
       2.98472238e-05, 2.98496640e-05, 2.98500000e-05]), fs=1000.0000000236469, dt=0.0009999999999763531, report=SamplingReport(n_samples=499738, duration_s=499.99811100000005, fs_nominal=1000.0000000236469, fs_regression=999.394899856747, dt_mean=0.0010005224968333342, dt_median=0.0009999999999763531, dt_std=3.396144477404712e-05, jitter_rms=3.3965429868216765e-05, jitter_ppm_rms=33965.429869019936, max_abs_jitter=0.002596000000011145, pct_within_tolerance=48.21316012222429, monotonic_non_decreasing=True, duplicate_timestamps=0, negative_steps=0, estimated_missing_intervals=261.11101182346465, uniform_enough=False, notes='Fewer than 95% of intervals meet the tolerance.'), resampled=True, info='Resampled to uniform grid using linear interpolation (dt≈0.001s).')


In [36]:
report = assess_uniform_sampling(ON_I_02_LPFno_ts, tolerance_ppm=1no)  # 0.15% RMS tolerance
print(report)


SamplingReport(n_samples=499646, duration_s=499998.119, fs_nominal=1.0, fs_regression=0.9995775159703044, dt_mean=1.0007067397852476, dt_median=1.0, dt_std=0.03529270271372783, jitter_rms=0.03529974296002622, jitter_ppm_rms=35299.74296002622, max_abs_jitter=1.8990000000001146, pct_within_tolerance=48.04851444525613, monotonic_non_decreasing=True, duplicate_timestamps=0, negative_steps=0, estimated_missing_intervals=353.11900000000605, uniform_enough=False, notes='Fewer than 95% of intervals meet the tolerance.')


In [36]:
def plot_resample_diagnostics(t, y, res, seconds=None, start=None, match_method="linear"):
    """
    Visual diagnostics for resampling.

    Parameters
    ----------
    t, y : original timestamps and values (1D, same length)
    res  : ResampleResult from resample_to_uniform()
    seconds : float or none
        Window length (seconds) to display. If none, use min(1.0 s, 5% of record).
    start : float or none
        Start time (absolute, same units as t). If none, use t[0].
    match_method : {"linear","zoh"}
        How to put the original data onto the uniform grid for residuals.
    """
    t = np.asarray(t, float).ravel()
    y = np.asarray(y, float).ravel()
    assert t.size == y.size, "t and y must have same length"

    # --- Decide plotting window ---
    rec_dur = float(t[-1] - t[0])
    if seconds is None:
        seconds = min(1.0, 0.05 * rec_dur)
        seconds = max(seconds, res.dt * 50)  # ensure at least ~50 samples
    if start is None:
        start = float(t[0])

    t0 = start
    t1 = min(start + seconds, t[-1])
    # Build masks for the chosen window
    m_raw = (t >= t0) & (t <= t1)
    m_uni = (res.t_uniform >= t0) & (res.t_uniform <= t1)

    # --- Put original data on the uniform grid for residuals ---
    if match_method.lower() == "linear":
        y_raw_on_uni = np.interp(res.t_uniform, t, y)
    elif match_method.lower() in ("zoh", "zero", "hold"):
        idx = np.searchsorted(t, res.t_uniform, side="right") - 1
        idx = np.clip(idx, 0, len(y) - 1)
        y_raw_on_uni = y[idx]
    else:
        raise ValueError("match_method must be 'linear' or 'zoh'")

    # Residual (only meaningful if we actually resampled)
    residual = res.y_uniform - y_raw_on_uni

    # ========== Figure 1: Overlay (zoomed window) ==========
    plt.figure()
    plt.plot(t[m_raw], y[m_raw], lw=1, label="Original")
    plt.plot(res.t_uniform[m_uni], res.y_uniform[m_uni], lw=1, alpha=0.8, label="Uniform")
    plt.xlabel("Time [s]")
    plt.ylabel("Signal")
    plt.title(f"Original vs Uniform (window {t0:.3f}–{t1:.3f}s)  |  {res.info}")
    plt.grid(True, which="both")
    plt.legend()

    # ========== Figure 2: Residual on uniform grid ==========
    plt.figure()
    plt.plot(res.t_uniform[m_uni], residual[m_uni], lw=1)
    plt.xlabel("Time [s]")
    plt.ylabel("Uniform − Original-on-Uniform")
    plt.title("Resampling residual on uniform grid")
    plt.grid(True, which="both")

    # ========== Figure 3: Δt histogram + nominal step ==========
    dt_raw = np.diff(t)
    plt.figure()
    plt.hist(dt_raw, bins="auto", alpha=0.8)
    plt.axvline(res.dt, linestyle="--", label=f"nominal dt ≈ {res.dt:.6g} s")
    plt.xlabel("Δt of original timestamps [s]")
    plt.ylabel("Count")
    plt.title("Timestamp spacing (Δt) histogram")
    plt.grid(True, which="both")
    plt.legend()

    # Print a quick textual summary to console
    r = res.report
    print(
        f"[Sampling] N={r.n_samples}, duration={r.duration_s:.6g}s, "
        f"fs_nominal={r.fs_nominal:.6g} Hz, fs_regression={r.fs_regression:.6g} Hz\n"
        f"Δt median={r.dt_median:.6g}s, RMS jitter={r.jitter_rms:.3g}s "
        f"({r.jitter_ppm_rms:.1f} ppm), within_tol={r.pct_within_tolerance:.1f}%, "
        f"uniform_enough={r.uniform_enough}\nnotes: {r.notes}"
    )

# ---------- Example (after you have `res` from resample_to_uniform) ----------
# result = resample_to_uniform(t, y, tolerance_ppm=no, method="linear")
# plot_resample_diagnostics(t, y, result, seconds=0.5)
# plt.show()

In [58]:
plot_resample_diagnostics(OFF_I_02_LPFno_ts,OFF_I_02_LPFno_xs, result)
plt.show()

[Sampling] N=499738, duration=499998s, fs_nominal=1 Hz, fs_regression=0.999395 Hz
Δt median=1s, RMS jitter=0.034s (33965.4 ppm), within_tol=48.2%, uniform_enough=False
Notes: Fewer than 95% of intervals meet the tolerance.


In [28]:
OFF_I_02_LPFno_fs=1/(1.0005224968333344*(10**(-3)))

In [27]:
# --- Allan deviation for coupling-efficiency stability (FSM OFF vs ON) ---
def compute_oadev(x, fs, taus=None):
    """
    x   : 1D array (uniformly sampled scalar time series, e.g., coupling efficiency)
    fs  : sample rate [Hz]
    taus: optional array of tau values [s]; if None, use log-spaced defaults
    returns: taus, adev, adeverr, n_pairs
    """
    x = np.asarray(x, dtype=float)
    # remove NaNs/Infs (keep uniform sampling!)
    good = np.isfinite(x)
    x = x[good]
    if x.size < 100:
        raise ValueError("Time series too short for Allan deviation (need at least ~100 samples).")
    if taus is None:
        # Use ~10 points/decade, from 1/fs to (N/3)/fs (enough for overlapping ADEV)
        m_min = 1
        m_max = max(2, x.size // 3)
        ms = np.unique(np.logspace(np.log10(m_min), np.log10(m_max), 
                                   int(10 * np.log10(m_max / m_min))).astype(int))
        taus = ms / fs
    # Overlapping Allan deviation
    taus_out, adev, adeverr, n = allantools.oadev(
        x, rate=fs, data_type="freq", taus=taus
    )
    return np.array(taus_out), np.array(adev), np.array(adeverr), np.array(n)

def plot_oadev_compare(taus, adev_off, adev_on, adeverr_off=None, adeverr_on=None):
    plt.figure()
    plt.loglog(taus, adev_off, label="FSM OFF")
    plt.loglog(taus, adev_on,  label="FSM ON")
    if adeverr_off is not None:
        plt.fill_between(taus, np.maximum(adev_off - adeverr_off, 0), adev_off + adeverr_off, alpha=0.2)
    if adeverr_on is not None:
        plt.fill_between(taus, np.maximum(adev_on  - adeverr_on,  0), adev_on  + adeverr_on,  alpha=0.2)
    plt.xlabel("Averaging time τ [s]")
    plt.ylabel("Overlapping Allan deviation σ(τ)")
    plt.grid(True, which="both")
    plt.legend()
    plt.title("Coupling-efficiency stability (Allan deviation)")

    # Improvement factor vs τ (σ_off / σ_on)
    plt.figure()
    improvement = adev_off / adev_on
    plt.semilogx(taus, improvement)
    plt.xlabel("Averaging time τ [s]")
    plt.ylabel("Stability improvement (σ_off / σ_on)")
    plt.grid(True, which="both")
    plt.title("FSM improvement vs τ")
    plt.show()

In [40]:
# --- Option A: start from an existing ResampleResult `res` ---
def compute_oadev_from_res(res, taus=None, use_hadamard=False):
    """
    Compute overlapping Allan deviation (or Hadamard) from a ResampleResult.

    Parameters
    ----------
    res : ResampleResult
        Output of resample_to_uniform(...). Must contain t_uniform, y_uniform, fs.
    taus : array-like or none
        Averaging times [s]. If none, a reasonable log-spaced grid is generated.
    use_hadamard : bool
        If True, compute overlapping Hadamard deviation (ohdev) instead of oadev.

    Returns
    -------
    taus_out, dev, dev_err, n, meta
        meta: dict with { 'fs': res.fs, 'dt': res.dt, 'resampled': res.resampled, 'info': res.info, 'report': res.report }
    """
    y = np.asarray(res.y_uniform, float).ravel()
    fs = float(res.fs)

    if taus is None:
        # ~10 pts/decade, τ from 1/fs to (N/3)/fs as long as N is large enough
        N = y.size
        m_min = 1
        m_max = max(2, N // 3)
        ms = np.unique(np.logspace(np.log10(m_min), np.log10(m_max),
                                   int(10 * max(1, np.log10(m_max / m_min)))).astype(int))
        taus = ms / fs
    else:
        taus = np.asarray(taus, float)

    if use_hadamard:
        taus_out, dev, dev_err, n = allantools.ohdev(y, rate=fs, data_type="freq", taus=taus)
    else:
        taus_out, dev, dev_err, n = allantools.oadev(y, rate=fs, data_type="freq", taus=taus)

    meta = {
        "fs": fs,
        "dt": res.dt,
        "resampled": res.resampled,
        "info": res.info,
        "report": res.report,  # your SamplingReport for diagnostics
    }
    return np.array(taus_out), np.array(dev), np.array(dev_err), np.array(n), meta


# --- Option B: start from raw timestamps & values; resample then compute ---
def compute_oadev_from_series(timestamps, values, tolerance_ppm=1000.0, method="linear",
                              taus=None, use_hadamard=False, snap_if_uniform=False, trim_end=True):
    """
    Convenience wrapper: resample_to_uniform(...) then compute Allan deviation.

    Returns
    -------
    taus_out, dev, dev_err, n, meta
        meta includes `res` (ResampleResult) so you can inspect/plot later.
    """
    # assumes you already defined resample_to_uniform(...) previously
    res = resample_to_uniform(
        timestamps, values,
        tolerance_ppm=tolerance_ppm,
        method=method,
        snap_if_uniform=snap_if_uniform,
        trim_end=trim_end,
    )
    taus_out, dev, dev_err, n, meta = compute_oadev_from_res(res, taus=taus, use_hadamard=use_hadamard)
    meta["res"] = res
    return taus_out, dev, dev_err, n, meta


In [ ]:
OFF_I_02_LPFno_ts, OFF_I_02_LPFno_xs
ON_I_02_LPFno_ts, ON_I_02_LPFno_xs

In [100]:
# OFF data
taus, adev_off, err_off, _, meta_off = compute_oadev_from_series(OFF_I_02_LPFNO_ts, OFF_I_02_LPFNO_xs, tolerance_ppm=500)

# ON data: resample separately, then compute with same taus
res_on = resample_to_uniform(ON_I_02_LPFNO_ts, ON_I_02_LPFNO_xs, tolerance_ppm=500, method="linear")
_, adev_on, err_on, _, meta_on = compute_oadev_from_res(res_on, taus=taus)

In [42]:
def plot_oadev_compare2(taus, adev_off, adev_on, adeverr_off=None, adeverr_on=None):
    """
    Compare Allan deviation (OFF vs ON) and highlight where FSM improves stability.

    Parameters
    ----------
    taus : array
        Averaging times [s]
    adev_off : array
        Allan deviation with FSM OFF
    adev_on : array
        Allan deviation with FSM ON
    adeverr_off, adeverr_on : array or none
        1-sigma errors (optional, same length as adevs)
    """
    taus = np.asarray(taus)
    adev_off = np.asarray(adev_off)
    adev_on = np.asarray(adev_on)

    # --- Identify where there is improvement ---
    improvement_mask = adev_off > adev_on
    no_improvement_mask = ~improvement_mask
    ratio = adev_off / adev_on

    # --- Figure 1: σ(τ) with shaded regions ---
    plt.figure(figsize=(7,5))
    plt.loglog(taus, adev_off, label="FSM OFF", color="C0")
    plt.loglog(taus, adev_on,  label="FSM ON",  color="C1")
    if adeverr_off is not None:
        plt.fill_between(taus, np.maximum(adev_off - adeverr_off, 0),
                         adev_off + adeverr_off, alpha=0.2, color="C0")
    if adeverr_on is not None:
        plt.fill_between(taus, np.maximum(adev_on - adeverr_on, 0),
                         adev_on + adeverr_on, alpha=0.2, color="C1")

    # Shading where FSM improves (OFF > ON) vs not
    plt.fill_between(taus, adev_on, adev_off, where=improvement_mask,
                     color="green", alpha=0.2, label="Improvement (OFF > ON)")
    plt.fill_between(taus, adev_on, adev_off, where=no_improvement_mask,
                     color="red", alpha=0.2, label="no improvement (OFF ≤ ON)")

    plt.xlabel("Averaging time τ [s]")
    plt.ylabel("Overlapping Allan deviation σ(τ)")
    plt.title("Coupling stability with FSM OFF vs ON")
    plt.grid(True, which="both")
    plt.legend()

    # --- Figure 2: Improvement ratio σ_off / σ_on ---
    plt.figure(figsize=(7,4))
    plt.semilogx(taus, ratio, color="k")
    plt.axhline(1.0, color="gray", linestyle="--")
    plt.fill_between(taus, ratio, 1, where=ratio>1, color="green", alpha=0.2, label="Improvement")
    plt.fill_between(taus, ratio, 1, where=ratio<=1, color="red", alpha=0.2, label="no improvement")
    plt.xlabel("Averaging time τ [s]")
    plt.ylabel("Improvement factor (σ_off / σ_on)")
    plt.title("Stability improvement vs τ")
    plt.grid(True, which="both")
    plt.legend()

    plt.show()


In [91]:
OFF_I_05_LPF500_ts, OFF_I_05_LPF500_xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-19\PID_1_05_0_OFF_LPF500_19_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
OFF_I_05_LPF500_ts=OFF_I_05_LPF500_ts*(10**(-3))

In [93]:
ON_I_05_LPF500_ts, ON_I_05_LPF500_xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-19\PID_1_05_0_ON_LPF500_19_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ON_I_05_LPF500_ts=ON_I_05_LPF500_ts*(10**(-3))

In [102]:
# OFF data
taus2, adev_off2, err_off2, _, meta_off2 = compute_oadev_from_series(OFF_I_05_LPF500_ts, OFF_I_05_LPF500_xs, tolerance_ppm=500)

# ON data: resample separately, then compute with same taus
res_on2 = resample_to_uniform(ON_I_05_LPF500_ts, ON_I_05_LPF500_xs, tolerance_ppm=500, method="linear")
_, adev_on2, err_on2, _, meta_on = compute_oadev_from_res(res_on2, taus=taus2)


In [528]:
plot_oadev_compare2(taus2, adev_off2, adev_on2, adeverr_off=None, adeverr_on=None)
plot_oadev_compare2(taus, adev_off, adev_on, adeverr_off=None, adeverr_on=None)

In [530]:
FSM_InitialImprovement_19_06_25=[(taus,adev_off),
                                 (taus,adev_on),
                                 (taus2,adev_off2),
                                 (taus2,adev_on2)]

In [532]:
np.save('FSM_InitialImprovement_19_06_25',FSM_InitialImprovement_19_06_25)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.loglog(taus, adev_off, label="FSM OFF")
plt.loglog(taus, adev_on,  label="FSM ON")
plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [22]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_0_0_0_OFF_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_OFF_26_06_25_ts=ts
I_OFF_26_06_25_xs=xs

In [24]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_0_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_00_26_06_25_ts=ts
I_00_26_06_25_xs=xs

In [117]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_01_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_01_26_06_25_ts=ts
I_01_26_06_25_xs=xs

In [119]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_02_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_02_26_06_25_ts=ts
I_02_26_06_25_xs=xs

In [121]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_03_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_03_26_06_25_ts=ts
I_03_26_06_25_xs=xs

In [123]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_04_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_04_26_06_25_ts=ts
I_04_26_06_25_xs=xs

In [125]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_05_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_05_26_06_25_ts=ts
I_05_26_06_25_xs=xs

In [127]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_06_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_06_26_06_25_ts=ts
I_06_26_06_25_xs=xs

In [129]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_07_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_07_26_06_25_ts=ts
I_07_26_06_25_xs=xs

In [131]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_08_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_08_26_06_25_ts=ts
I_08_26_06_25_xs=xs

In [133]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_09_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_09_26_06_25_ts=ts
I_09_26_06_25_xs=xs

In [135]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-26\PID_1_1_0_ON_LPFNO_26_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_10_26_06_25_ts=ts
I_10_26_06_25_xs=xs

In [44]:
taus_26_06_25, adev_OFF_26_06_25, err_OFF_26_06_25, _, meta_OFF_26_06_25 = compute_oadev_from_series(I_OFF_26_06_25_ts, I_OFF_26_06_25_xs, tolerance_ppm=500)

In [46]:
np.save('taus_26_06_25.npy',taus_26_06_25)

In [48]:
res_I_00_26_06_25 = resample_to_uniform(I_00_26_06_25_ts, I_00_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_00_26_06_25, err_I_00_26_06_25, _, meta_on = compute_oadev_from_res(res_I_00_26_06_25, taus=taus_26_06_25)

In [145]:
res_I_01_26_06_25 = resample_to_uniform(I_01_26_06_25_ts, I_01_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_01_26_06_25, err_I_01_26_06_25, _, meta_on = compute_oadev_from_res(res_I_01_26_06_25, taus=taus_26_06_25)

In [147]:
res_I_02_26_06_25 = resample_to_uniform(I_02_26_06_25_ts, I_02_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_02_26_06_25, err_I_02_26_06_25, _, meta_on = compute_oadev_from_res(res_I_02_26_06_25, taus=taus_26_06_25)

In [149]:
res_I_03_26_06_25 = resample_to_uniform(I_03_26_06_25_ts, I_03_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_03_26_06_25, err_I_03_26_06_25, _, meta_on = compute_oadev_from_res(res_I_03_26_06_25, taus=taus_26_06_25)

In [151]:
res_I_04_26_06_25 = resample_to_uniform(I_04_26_06_25_ts, I_04_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_04_26_06_25, err_I_04_26_06_25, _, meta_on = compute_oadev_from_res(res_I_04_26_06_25, taus=taus_26_06_25)

In [153]:
res_I_05_26_06_25 = resample_to_uniform(I_05_26_06_25_ts, I_05_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_05_26_06_25, err_I_05_26_06_25, _, meta_on = compute_oadev_from_res(res_I_05_26_06_25, taus=taus_26_06_25)

In [155]:
res_I_06_26_06_25 = resample_to_uniform(I_06_26_06_25_ts, I_06_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_06_26_06_25, err_I_06_26_06_25, _, meta_on = compute_oadev_from_res(res_I_06_26_06_25, taus=taus_26_06_25)

In [157]:
res_I_07_26_06_25 = resample_to_uniform(I_07_26_06_25_ts, I_07_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_07_26_06_25, err_I_07_26_06_25, _, meta_on = compute_oadev_from_res(res_I_07_26_06_25, taus=taus_26_06_25)

In [159]:
res_I_08_26_06_25 = resample_to_uniform(I_08_26_06_25_ts, I_08_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_08_26_06_25, err_I_08_26_06_25, _, meta_on = compute_oadev_from_res(res_I_08_26_06_25, taus=taus_26_06_25)

In [161]:
res_I_09_26_06_25 = resample_to_uniform(I_09_26_06_25_ts, I_09_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_09_26_06_25, err_I_09_26_06_25, _, meta_on = compute_oadev_from_res(res_I_09_26_06_25, taus=taus_26_06_25)

In [163]:
res_I_10_26_06_25 = resample_to_uniform(I_10_26_06_25_ts, I_10_26_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_10_26_06_25, err_I_10_26_06_25, _, meta_on = compute_oadev_from_res(res_I_10_26_06_25, taus=taus_26_06_25)

In [51]:
plt.figure()
plt.loglog(taus_26_06_25, adev_OFF_26_06_25, label="FSM OFF")
plt.loglog(taus_26_06_25, adev_I_00_26_06_25,  label="FSM I = 00")
#plt.loglog(taus_26_06_25, adev_I_01_26_06_25,  label="FSM I = 01")
#plt.loglog(taus_26_06_25, adev_I_02_26_06_25,  label="FSM I = 02")
#plt.loglog(taus_26_06_25, adev_I_03_26_06_25,  label="FSM I = 03")
#plt.loglog(taus_26_06_25, adev_I_04_26_06_25,  label="FSM I = 04")
#plt.loglog(taus_26_06_25, adev_I_05_26_06_25,  label="FSM I = 05")
#plt.loglog(taus_26_06_25, adev_I_06_26_06_25,  label="FSM I = 06")
#plt.loglog(taus_26_06_25, adev_I_07_26_06_25,  label="FSM I = 07")
#plt.loglog(taus_26_06_25, adev_I_08_26_06_25,  label="FSM I = 08")
#plt.loglog(taus_26_06_25, adev_I_09_26_06_25,  label="FSM I = 09")
#plt.loglog(taus_26_06_25, adev_I_10_26_06_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [524]:
FSM_Icomparison_26_06_25=[(taus_26_06_25, adev_OFF_26_06_25),
                          (taus_26_06_25, adev_I_00_26_06_25),
                          (taus_26_06_25, adev_I_01_26_06_25),
                          (taus_26_06_25, adev_I_02_26_06_25),
                          (taus_26_06_25, adev_I_03_26_06_25),
                          (taus_26_06_25, adev_I_04_26_06_25),
                          (taus_26_06_25, adev_I_05_26_06_25),
                          (taus_26_06_25, adev_I_06_26_06_25),
                          (taus_26_06_25, adev_I_07_26_06_25),
                          (taus_26_06_25, adev_I_08_26_06_25),
                          (taus_26_06_25, adev_I_09_26_06_25),
                          (taus_26_06_25, adev_I_10_26_06_25)]

In [526]:
np.save('FSM_Icomparison_26_06_25',FSM_Icomparison_26_06_25)

In [269]:
scale=adev_OFF_26_06_25

plt.figure()
plt.loglog(taus_26_06_25, scale/adev_OFF_26_06_25, label="FSM OFF")
plt.loglog(taus_26_06_25, scale/adev_I_00_26_06_25,  label="FSM I = 00")
plt.loglog(taus_26_06_25, scale/adev_I_01_26_06_25,  label="FSM I = 01")
plt.loglog(taus_26_06_25, scale/adev_I_02_26_06_25,  label="FSM I = 02")
plt.loglog(taus_26_06_25, scale/adev_I_03_26_06_25,  label="FSM I = 03")
plt.loglog(taus_26_06_25, scale/adev_I_04_26_06_25,  label="FSM I = 04")
plt.loglog(taus_26_06_25, scale/adev_I_05_26_06_25,  label="FSM I = 05")
plt.loglog(taus_26_06_25, scale/adev_I_06_26_06_25,  label="FSM I = 06")
plt.loglog(taus_26_06_25, scale/adev_I_07_26_06_25,  label="FSM I = 07")
plt.loglog(taus_26_06_25, scale/adev_I_08_26_06_25,  label="FSM I = 08")
plt.loglog(taus_26_06_25, scale/adev_I_09_26_06_25,  label="FSM I = 09")
plt.loglog(taus_26_06_25, scale/adev_I_10_26_06_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [193]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-24\PID_1_04_0_OFF_LPFNO_24_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_OFF_24_06_25_ts=ts
I_OFF_24_06_25_xs=xs

In [199]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-06-24\PID_1_04_0_ON_LPFNO_24_06_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_04_24_06_25_ts=ts
I_04_24_06_25_xs=xs

In [207]:
taus_24_06_25, adev_OFF_24_06_25, err_OFF_24_06_25, _, meta_OFF_24_06_25 = compute_oadev_from_series(I_OFF_24_06_25_ts, I_OFF_24_06_25_xs, tolerance_ppm=500)

In [209]:
res_I_04_24_06_25 = resample_to_uniform(I_04_24_06_25_ts, I_04_24_06_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_04_24_06_25, err_I_04_24_06_25, _, meta_on = compute_oadev_from_res(res_I_04_24_06_25, taus=taus_24_06_25)

In [211]:
plot_oadev_compare2(taus_24_06_25, adev_OFF_24_06_25, adev_I_04_24_06_25, adeverr_off=None, adeverr_on=None)

In [534]:
FSM_SecondTest_24_06_25=[(taus_24_06_25,adev_OFF_24_06_25),
                         (taus_24_06_25,adev_I_04_24_06_25)]

In [536]:
np.save('FSM_SecondTest_24_06_25',FSM_SecondTest_24_06_25)

In [213]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_0_0_0_OFF_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_OFF_02_07_25_ts=ts
I_OFF_02_07_25_xs=xs

In [215]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_00_02_07_25_ts=ts
I_00_02_07_25_xs=xs

In [217]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0001_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_01_02_07_25_ts=ts
I_01_02_07_25_xs=xs

In [219]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0002_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_02_02_07_25_ts=ts
I_02_02_07_25_xs=xs

In [221]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0003_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_03_02_07_25_ts=ts
I_03_02_07_25_xs=xs

In [223]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0004_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_04_02_07_25_ts=ts
I_04_02_07_25_xs=xs

In [225]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0005_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_05_02_07_25_ts=ts
I_05_02_07_25_xs=xs

In [227]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0006_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_06_02_07_25_ts=ts
I_06_02_07_25_xs=xs

In [229]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0007_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_07_02_07_25_ts=ts
I_07_02_07_25_xs=xs

In [231]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0008_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_08_02_07_25_ts=ts
I_08_02_07_25_xs=xs

In [233]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_0009_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_09_02_07_25_ts=ts
I_09_02_07_25_xs=xs

In [235]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-02\PID_1_001_0_ON_LPFNO_02_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_10_02_07_25_ts=ts
I_10_02_07_25_xs=xs

In [237]:
taus_02_07_25, adev_OFF_02_07_25, err_OFF_02_07_25, _, meta_OFF_02_07_25 = compute_oadev_from_series(I_OFF_02_07_25_ts, I_OFF_02_07_25_xs, tolerance_ppm=500)

In [239]:
res_I_00_02_07_25 = resample_to_uniform(I_00_02_07_25_ts, I_00_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_00_02_07_25, err_I_00_02_07_25, _, meta_on = compute_oadev_from_res(res_I_00_02_07_25, taus=taus_02_07_25)

In [241]:
res_I_01_02_07_25 = resample_to_uniform(I_01_02_07_25_ts, I_01_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_01_02_07_25, err_I_01_02_07_25, _, meta_on = compute_oadev_from_res(res_I_01_02_07_25, taus=taus_02_07_25)

In [243]:
res_I_02_02_07_25 = resample_to_uniform(I_02_02_07_25_ts, I_02_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_02_02_07_25, err_I_02_02_07_25, _, meta_on = compute_oadev_from_res(res_I_02_02_07_25, taus=taus_02_07_25)

In [245]:
res_I_03_02_07_25 = resample_to_uniform(I_03_02_07_25_ts, I_03_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_03_02_07_25, err_I_03_02_07_25, _, meta_on = compute_oadev_from_res(res_I_03_02_07_25, taus=taus_02_07_25)

In [247]:
res_I_04_02_07_25 = resample_to_uniform(I_04_02_07_25_ts, I_04_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_04_02_07_25, err_I_04_02_07_25, _, meta_on = compute_oadev_from_res(res_I_04_02_07_25, taus=taus_02_07_25)

In [249]:
res_I_05_02_07_25 = resample_to_uniform(I_05_02_07_25_ts, I_05_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_05_02_07_25, err_I_05_02_07_25, _, meta_on = compute_oadev_from_res(res_I_05_02_07_25, taus=taus_02_07_25)

In [251]:
res_I_06_02_07_25 = resample_to_uniform(I_06_02_07_25_ts, I_06_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_06_02_07_25, err_I_06_02_07_25, _, meta_on = compute_oadev_from_res(res_I_06_02_07_25, taus=taus_02_07_25)

In [253]:
res_I_07_02_07_25 = resample_to_uniform(I_07_02_07_25_ts, I_07_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_07_02_07_25, err_I_07_02_07_25, _, meta_on = compute_oadev_from_res(res_I_07_02_07_25, taus=taus_02_07_25)

In [255]:
res_I_08_02_07_25 = resample_to_uniform(I_08_02_07_25_ts, I_08_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_08_02_07_25, err_I_08_02_07_25, _, meta_on = compute_oadev_from_res(res_I_08_02_07_25, taus=taus_02_07_25)

In [257]:
res_I_09_02_07_25 = resample_to_uniform(I_09_02_07_25_ts, I_09_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_09_02_07_25, err_I_09_02_07_25, _, meta_on = compute_oadev_from_res(res_I_09_02_07_25, taus=taus_02_07_25)

In [259]:
res_I_10_02_07_25 = resample_to_uniform(I_10_02_07_25_ts, I_10_02_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_10_02_07_25, err_I_10_02_07_25, _, meta_on = compute_oadev_from_res(res_I_10_02_07_25, taus=taus_02_07_25)

In [273]:
plt.figure()
plt.loglog(taus_02_07_25, adev_OFF_02_07_25, label="FSM OFF")
plt.loglog(taus_02_07_25, adev_I_00_02_07_25,  label="FSM I = 00")
plt.loglog(taus_02_07_25, adev_I_01_02_07_25,  label="FSM I = 01")
plt.loglog(taus_02_07_25, adev_I_02_02_07_25,  label="FSM I = 02")
plt.loglog(taus_02_07_25, adev_I_03_02_07_25,  label="FSM I = 03")
plt.loglog(taus_02_07_25, adev_I_04_02_07_25,  label="FSM I = 04")
plt.loglog(taus_02_07_25, adev_I_05_02_07_25,  label="FSM I = 05")
plt.loglog(taus_02_07_25, adev_I_06_02_07_25,  label="FSM I = 06")
plt.loglog(taus_02_07_25, adev_I_07_02_07_25,  label="FSM I = 07")
plt.loglog(taus_02_07_25, adev_I_08_02_07_25,  label="FSM I = 08")
plt.loglog(taus_02_07_25, adev_I_09_02_07_25,  label="FSM I = 09")
plt.loglog(taus_02_07_25, adev_I_10_02_07_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [300]:
plt.figure()
plt.loglog(taus_02_07_25, err_OFF_02_07_25, label="FSM OFF")
plt.loglog(taus_02_07_25, err_I_00_02_07_25,  label="FSM I = 00")
plt.loglog(taus_02_07_25, err_I_01_02_07_25,  label="FSM I = 01")
plt.loglog(taus_02_07_25, err_I_02_02_07_25,  label="FSM I = 02")
plt.loglog(taus_02_07_25, err_I_03_02_07_25,  label="FSM I = 03")
plt.loglog(taus_02_07_25, err_I_04_02_07_25,  label="FSM I = 04")
plt.loglog(taus_02_07_25, err_I_05_02_07_25,  label="FSM I = 05")
plt.loglog(taus_02_07_25, err_I_06_02_07_25,  label="FSM I = 06")
plt.loglog(taus_02_07_25, err_I_07_02_07_25,  label="FSM I = 07")
plt.loglog(taus_02_07_25, err_I_08_02_07_25,  label="FSM I = 08")
plt.loglog(taus_02_07_25, err_I_09_02_07_25,  label="FSM I = 09")
plt.loglog(taus_02_07_25, err_I_10_02_07_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [271]:
scale=adev_OFF_02_07_25

plt.figure()
plt.loglog(taus_02_07_25, scale/adev_OFF_02_07_25, label="FSM OFF")
plt.loglog(taus_02_07_25, scale/adev_I_00_02_07_25,  label="FSM I = 00")
plt.loglog(taus_02_07_25, scale/adev_I_01_02_07_25,  label="FSM I = 01")
plt.loglog(taus_02_07_25, scale/adev_I_02_02_07_25,  label="FSM I = 02")
plt.loglog(taus_02_07_25, scale/adev_I_03_02_07_25,  label="FSM I = 03")
plt.loglog(taus_02_07_25, scale/adev_I_04_02_07_25,  label="FSM I = 04")
plt.loglog(taus_02_07_25, scale/adev_I_05_02_07_25,  label="FSM I = 05")
plt.loglog(taus_02_07_25, scale/adev_I_06_02_07_25,  label="FSM I = 06")
plt.loglog(taus_02_07_25, scale/adev_I_07_02_07_25,  label="FSM I = 07")
plt.loglog(taus_02_07_25, scale/adev_I_08_02_07_25,  label="FSM I = 08")
plt.loglog(taus_02_07_25, scale/adev_I_09_02_07_25,  label="FSM I = 09")
plt.loglog(taus_02_07_25, scale/adev_I_10_02_07_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [302]:
scale=err_OFF_02_07_25

plt.figure()
plt.loglog(taus_02_07_25, scale/err_OFF_02_07_25, label="FSM OFF")
plt.loglog(taus_02_07_25, scale/err_I_00_02_07_25,  label="FSM I = 00")
plt.loglog(taus_02_07_25, scale/err_I_01_02_07_25,  label="FSM I = 01")
plt.loglog(taus_02_07_25, scale/err_I_02_02_07_25,  label="FSM I = 02")
plt.loglog(taus_02_07_25, scale/err_I_03_02_07_25,  label="FSM I = 03")
plt.loglog(taus_02_07_25, scale/err_I_04_02_07_25,  label="FSM I = 04")
plt.loglog(taus_02_07_25, scale/err_I_05_02_07_25,  label="FSM I = 05")
plt.loglog(taus_02_07_25, scale/err_I_06_02_07_25,  label="FSM I = 06")
plt.loglog(taus_02_07_25, scale/err_I_07_02_07_25,  label="FSM I = 07")
plt.loglog(taus_02_07_25, scale/err_I_08_02_07_25,  label="FSM I = 08")
plt.loglog(taus_02_07_25, scale/err_I_09_02_07_25,  label="FSM I = 09")
plt.loglog(taus_02_07_25, scale/err_I_10_02_07_25,  label="FSM I = 10")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [520]:
FSM_Icomparison_02_07_25=[(taus_02_07_25, adev_OFF_02_07_25),
                          (taus_02_07_25, adev_I_00_02_07_25),
                          (taus_02_07_25, adev_I_01_02_07_25),
                          (taus_02_07_25, adev_I_02_02_07_25),
                          (taus_02_07_25, adev_I_03_02_07_25),
                          (taus_02_07_25, adev_I_04_02_07_25),
                          (taus_02_07_25, adev_I_05_02_07_25),
                          (taus_02_07_25, adev_I_06_02_07_25),
                          (taus_02_07_25, adev_I_07_02_07_25),
                          (taus_02_07_25, adev_I_08_02_07_25),
                          (taus_02_07_25, adev_I_09_02_07_25),
                          (taus_02_07_25, adev_I_10_02_07_25)]

In [522]:
np.save('FSM_Icomparison_02_07_25',FSM_Icomparison_02_07_25)

In [288]:
import numpy as np
import matplotlib.pyplot as plt

def _log_interp(x, xp, fp):
    x  = np.asarray(x,  float)
    xp = np.asarray(xp, float)
    fp = np.asarray(fp, float)
    if np.any(x <= 0) or np.any(xp <= 0) or np.any(fp <= 0):
        return np.interp(x, xp, fp, left=np.nan, right=np.nan)
    return np.exp(np.interp(np.log(x), np.log(xp), np.log(fp), left=np.nan, right=np.nan))

def plot_multi_adev(
    series,
    title="Allan deviation comparison",
    baseline_label=None,
    show_ratio=True,
    points_per_decade=24,
    err_alpha=0.12,
    linewidth=1.6,
    envelope_linewidth=2.4,
    legend_loc="right",          # "right", "bottom", or "inside"
    legend_ncol=None,            # auto if None
    legend_fontsize=9,
    figsize=(8.5, 6.8),
):
    # ---- validate & normalize input ----
    if not isinstance(series, (list, tuple)) or len(series) < 2:
        raise ValueError("Provide at least two series to compare.")
    clean = []
    for s in series:
        taus = np.asarray(s["taus"], float).ravel()
        adev = np.asarray(s["adev"], float).ravel()
        if taus.size != adev.size:
            raise ValueError(f"taus and adev must match for '{s.get('label','?')}'")
        order = np.argsort(taus)
        item = {
            "taus": taus[order],
            "adev": adev[order],
            "label": str(s.get("label", f"Series {len(clean)+1}"))
        }
        if "err" in s and s["err"] is not None:
            err = np.asarray(s["err"], float).ravel()[order]
            if err.size != item["taus"].size:
                raise ValueError(f"err must match length for '{item['label']}'")
            item["err"] = err
        clean.append(item)
    series = clean

    tau_min = min(np.min(s["taus"]) for s in series)
    tau_max = max(np.max(s["taus"]) for s in series)
    if tau_max <= tau_min:
        raise ValueError("Degenerate τ ranges.")
    ndec = np.log10(tau_max) - np.log10(tau_min)
    npts = max(16, int(points_per_decade * max(1.0, ndec)))
    tau_common = np.logspace(np.log10(tau_min), np.log10(tau_max), npts)

    A_common = np.vstack([_log_interp(tau_common, s["taus"], s["adev"]) for s in series])
    envelope = np.nanmin(A_common, axis=0)

    # baseline selection
    baseline = None
    if show_ratio and baseline_label is not None:
        baseline = next((s for s in series if s["label"] == baseline_label), None)
        if baseline is None:
            raise ValueError(f"baseline_label='{baseline_label}' not found.")
    elif show_ratio:
        baseline = series[0]
        baseline_label = baseline["label"]

    # figure/axes
    if show_ratio:
        fig = plt.figure(figsize=figsize, constrained_layout=False)
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[2.4, 1.0], hspace=0.22)
        ax_top = fig.add_subplot(gs[0, 0])
        ax_bot = fig.add_subplot(gs[1, 0], sharex=ax_top)
    else:
        fig, ax_top = plt.subplots(figsize=figsize, constrained_layout=False)
        ax_bot = None

    # plot σ(τ)
    handles = []
    labels = []
    for s in series:
        ln, = ax_top.loglog(s["taus"], s["adev"], linewidth=linewidth, label=s["label"])
        handles.append(ln)
        labels.append(s["label"])
        if "err" in s:
            lo = np.maximum(s["adev"] - s["err"], 0)
            hi = s["adev"] + s["err"]
            ax_top.fill_between(s["taus"], lo, hi, alpha=err_alpha)

    # envelope
    ln_env, = ax_top.loglog(tau_common, envelope, linewidth=envelope_linewidth, color="k", alpha=0.85)
    handles.append(ln_env)
    labels.append("Best envelope")

    ax_top.set_title(title)
    ax_top.set_ylabel("Allan deviation σ(τ)")
    ax_top.grid(True, which="both")

    # ratio panel
    if ax_bot is not None:
        base_on = _log_interp(tau_common, baseline["taus"], baseline["adev"])
        for s in series:
            s_on = _log_interp(tau_common, s["taus"], s["adev"])
            ratio = base_on/s_on
            ax_bot.semilogx(tau_common, ratio, linewidth=linewidth)
        ax_bot.axhline(1.0, linestyle="--", color="gray", linewidth=1)
        ax_bot.set_xlabel("Averaging time τ [s]")
        ax_bot.set_ylabel(f"σ / σ({baseline_label})")
        ax_bot.grid(True, which="both")

    # -------- robust legend placement --------
    n_items = len(labels)
    if legend_ncol is None:
        legend_ncol = 1 if n_items <= 6 else 2 if n_items <= 12 else 3

    if legend_loc == "right":
        # reserve right margin and place legend there
        fig.subplots_adjust(right=0.78)  # widen right margin so legend is visible
        fig.legend(handles, labels, loc="center right", bbox_to_anchor=(0.98, 0.5),
                   frameon=True, ncol=legend_ncol, fontsize=legend_fontsize)
    elif legend_loc == "bottom":
        # reserve bottom margin and span both axes
        fig.subplots_adjust(bottom=0.22)
        fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.02),
                   frameon=True, ncol=max(2, legend_ncol), fontsize=legend_fontsize)
    elif legend_loc == "inside":
        ax_top.legend(handles, labels, loc="best", frameon=True, ncol=legend_ncol, fontsize=legend_fontsize)
    else:
        raise ValueError("legend_loc must be 'right', 'bottom', or 'inside'.")

    plt.show()

In [308]:
# Example: list of series dicts
series = [
    {"taus": taus_02_07_25, "adev": adev_OFF_02_07_25, "err": err_OFF_02_07_25, "label": "FSM OFF"},
    {"taus": taus_02_07_25, "adev": adev_I_00_02_07_25, "err": err_I_00_02_07_25, "label": "I_00"},
    {"taus": taus_02_07_25, "adev": adev_I_01_02_07_25, "err": err_I_01_02_07_25, "label": "I_01"},
    {"taus": taus_02_07_25, "adev": adev_I_02_02_07_25, "err": err_I_02_02_07_25, "label": "I_02"},
    {"taus": taus_02_07_25, "adev": adev_I_03_02_07_25, "err": err_I_03_02_07_25, "label": "I_03"},
    {"taus": taus_02_07_25, "adev": adev_I_04_02_07_25, "err": err_I_04_02_07_25, "label": "I_04"},
    {"taus": taus_02_07_25, "adev": adev_I_05_02_07_25, "err": err_I_05_02_07_25, "label": "I_05"},
    {"taus": taus_02_07_25, "adev": adev_I_06_02_07_25, "err": err_I_06_02_07_25, "label": "I_06"},
    {"taus": taus_02_07_25, "adev": adev_I_07_02_07_25, "err": err_I_07_02_07_25, "label": "I_07"},
    {"taus": taus_02_07_25, "adev": adev_I_08_02_07_25, "err": err_I_08_02_07_25, "label": "I_08"},
    {"taus": taus_02_07_25, "adev": adev_I_09_02_07_25, "err": err_I_09_02_07_25, "label": "I_09"},
    {"taus": taus_02_07_25, "adev": adev_I_10_02_07_25, "err": err_I_10_02_07_25, "label": "I_10"}
    # ... up to ~11 series
]

plot_multi_adev(
    series,                           # your list of {taus, adev, err?, label}
    title="Coupling stability – ADEV",
    baseline_label="FSM OFF",         # or omit to use the first series
    show_ratio=True,
    err_alpha=0.4,
    legend_loc="inside",              # <- try "bottom" or "inside" if "right" clips
    legend_ncol=1,                    # tune columns for many series
    legend_fontsize=12,                # smaller font helps with 10–11 entries
)


In [ ]:
v

In [311]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_001_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_01_04_07_25_ts=ts
I_01_04_07_25_xs=xs

In [313]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_002_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_02_04_07_25_ts=ts
I_02_04_07_25_xs=xs

In [315]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_003_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_03_04_07_25_ts=ts
I_03_04_07_25_xs=xs

In [317]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_004_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_04_04_07_25_ts=ts
I_04_04_07_25_xs=xs

In [319]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_005_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
I_05_04_07_25_ts=ts
I_05_04_07_25_xs=xs

In [321]:
res_I_01_04_07_25 = resample_to_uniform(I_01_04_07_25_ts, I_01_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_01_04_07_25, err_I_01_04_07_25, _, meta_on = compute_oadev_from_res(res_I_01_04_07_25, taus=taus_02_07_25)

In [323]:
res_I_02_04_07_25 = resample_to_uniform(I_02_04_07_25_ts, I_02_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_02_04_07_25, err_I_02_04_07_25, _, meta_on = compute_oadev_from_res(res_I_02_04_07_25, taus=taus_02_07_25)

In [325]:
res_I_03_04_07_25 = resample_to_uniform(I_03_04_07_25_ts, I_03_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_03_04_07_25, err_I_03_04_07_25, _, meta_on = compute_oadev_from_res(res_I_03_04_07_25, taus=taus_02_07_25)

In [327]:
res_I_04_04_07_25 = resample_to_uniform(I_04_04_07_25_ts, I_04_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_04_04_07_25, err_I_04_04_07_25, _, meta_on = compute_oadev_from_res(res_I_04_04_07_25, taus=taus_02_07_25)

In [329]:
res_I_05_04_07_25 = resample_to_uniform(I_05_04_07_25_ts, I_05_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_I_05_04_07_25, err_I_05_04_07_25, _, meta_on = compute_oadev_from_res(res_I_05_04_07_25, taus=taus_02_07_25)

In [347]:
# Example: list of series dicts
series2 = [
    {"taus": taus_02_07_25, "adev": adev_I_01_04_07_25, "err": err_I_01_04_07_25, "label": "I_01"},
    {"taus": taus_02_07_25, "adev": adev_I_02_04_07_25, "err": err_I_02_04_07_25, "label": "I_02"},
    {"taus": taus_02_07_25, "adev": adev_I_03_04_07_25, "err": err_I_03_04_07_25, "label": "I_03"},
    {"taus": taus_02_07_25, "adev": adev_I_04_04_07_25, "err": err_I_04_04_07_25, "label": "I_04"},
    {"taus": taus_02_07_25, "adev": adev_I_05_04_07_25, "err": err_I_05_04_07_25, "label": "I_05"},
    # ... up to ~11 series
]

plot_multi_adev(
    series2,                           # your list of {taus, adev, err?, label}
    title="Coupling stability – ADEV",
    baseline_label="I_01",         # or omit to use the first series
    show_ratio=True,
    err_alpha=0.4,
    legend_loc="inside",              # <- try "bottom" or "inside" if "right" clips
    legend_ncol=1,                    # tune columns for many series
    legend_fontsize=12,                # smaller font helps with 10–11 entries
)


In [345]:
plt.figure()
plt.loglog(taus_02_07_25, adev_I_01_04_07_25,  label="FSM I = 001")
plt.loglog(taus_02_07_25, adev_I_02_04_07_25,  label="FSM I = 002")
plt.loglog(taus_02_07_25, adev_I_03_04_07_25,  label="FSM I = 003")
plt.loglog(taus_02_07_25, adev_I_04_04_07_25,  label="FSM I = 004")
plt.loglog(taus_02_07_25, adev_I_05_04_07_25,  label="FSM I = 005")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [339]:
scale=adev_OFF_02_07_25

plt.figure()
plt.loglog(taus_02_07_25, scale/adev_I_01_04_07_25,  label="FSM I = 001")
plt.loglog(taus_02_07_25, scale/adev_I_02_04_07_25,  label="FSM I = 002")
plt.loglog(taus_02_07_25, scale/adev_I_03_04_07_25,  label="FSM I = 003")
plt.loglog(taus_02_07_25, scale/adev_I_04_04_07_25,  label="FSM I = 004")
plt.loglog(taus_02_07_25, scale/adev_I_05_04_07_25,  label="FSM I = 005")
plt.loglog(taus_02_07_25, scale/adev_I_09_02_07_25,  label="FSM I = 0009")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [516]:
FSM_Icomparison_04_07_25=[(taus_02_07_25, adev_I_01_04_07_25),
                          (taus_02_07_25, adev_I_02_04_07_25),
                          (taus_02_07_25, adev_I_03_04_07_25),
                          (taus_02_07_25, adev_I_04_04_07_25),
                          (taus_02_07_25, adev_I_05_04_07_25)]

In [518]:
np.save('FSM_Icomparison_04_07_25',FSM_Icomparison_04_07_25)

In [351]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_1_001_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
P_100_04_07_25_ts=ts
P_100_04_07_25_xs=xs

In [354]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_075_001_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
P_075_04_07_25_ts=ts
P_075_04_07_25_xs=xs

In [356]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-04\PID_05_001_0_ON_LPFNO_04_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
P_050_04_07_25_ts=ts
P_050_04_07_25_xs=xs

In [358]:
taus_P_100_04_07_25, adev_P_100_04_07_25, err_P_100_04_07_25, _, meta_P_100_04_07_25 = compute_oadev_from_series(P_100_04_07_25_ts, P_100_04_07_25_xs, tolerance_ppm=500)

In [362]:
res_P_075_04_07_25 = resample_to_uniform(P_075_04_07_25_ts, P_075_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_P_075_04_07_25, err_P_075_04_07_25, _, meta_on = compute_oadev_from_res(res_P_075_04_07_25, taus=taus_P_100_04_07_25)

In [364]:
res_P_050_04_07_25 = resample_to_uniform(P_050_04_07_25_ts, P_050_04_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_P_050_04_07_25, err_P_050_04_07_25, _, meta_on = compute_oadev_from_res(res_P_050_04_07_25, taus=taus_P_100_04_07_25)

In [366]:
plt.figure()
plt.loglog(taus_P_100_04_07_25, adev_P_100_04_07_25,  label="FSM P = 1")
plt.loglog(taus_P_100_04_07_25, adev_P_075_04_07_25,  label="FSM P = 0.75")
plt.loglog(taus_P_100_04_07_25, adev_P_050_04_07_25,  label="FSM P = 0.50")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [368]:
plt.figure()
scale=adev_P_100_04_07_25
plt.loglog(taus_P_100_04_07_25, scale/adev_P_100_04_07_25,  label="FSM P = 1")
plt.loglog(taus_P_100_04_07_25, scale/adev_P_075_04_07_25,  label="FSM P = 0.75")
plt.loglog(taus_P_100_04_07_25, scale/adev_P_050_04_07_25,  label="FSM P = 0.50")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [512]:
FSM_Pcomparison_04_07_25=[(taus_P_100_04_07_25, adev_P_100_04_07_25),
                          (taus_P_100_04_07_25, adev_P_075_04_07_25),
                          (taus_P_100_04_07_25, adev_P_050_04_07_25)]

In [514]:
np.save('FSM_Pcomparison_04_07_25',FSM_Pcomparison_04_07_25)

In [370]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_0_0_0_OFF_LPFNO_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
OFF_07_07_25_ts=ts
OFF_07_07_25_xs=xs

In [372]:
taus_OFF_07_07_25, adev_OFF_07_07_25, err_OFF_07_07_25, _, meta_OFF_07_07_25 = compute_oadev_from_series(OFF_07_07_25_ts, OFF_07_07_25_xs, tolerance_ppm=500)

In [374]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPF1000_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF1000_07_07_25_ts=ts
LPF1000_07_07_25_xs=xs

In [376]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPF750_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF750_07_07_25_ts=ts
LPF750_07_07_25_xs=xs

In [378]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPF500_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF500_07_07_25_ts=ts
LPF500_07_07_25_xs=xs

In [380]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPF250_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF250_07_07_25_ts=ts
LPF250_07_07_25_xs=xs

In [382]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPF50_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF50_07_07_25_ts=ts
LPF50_07_07_25_xs=xs

In [384]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_LPFNO_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPFNO_07_07_25_ts=ts
LPFNO_07_07_25_xs=xs

In [387]:
res_LPF1000_07_07_25 = resample_to_uniform(LPF1000_07_07_25_ts, LPF1000_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF1000_07_07_25, err_LPF1000_07_07_25, _, meta_on = compute_oadev_from_res(res_LPF1000_07_07_25, taus=taus_OFF_07_07_25)

In [389]:
res_LPF750_07_07_25 = resample_to_uniform(LPF750_07_07_25_ts, LPF750_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF750_07_07_25, err_LPF750_07_07_25, _, meta_on = compute_oadev_from_res(res_LPF750_07_07_25, taus=taus_OFF_07_07_25)

In [391]:
res_LPF500_07_07_25 = resample_to_uniform(LPF500_07_07_25_ts, LPF500_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF500_07_07_25, err_LPF500_07_07_25, _, meta_on = compute_oadev_from_res(res_LPF500_07_07_25, taus=taus_OFF_07_07_25)

In [393]:
res_LPF250_07_07_25 = resample_to_uniform(LPF250_07_07_25_ts, LPF250_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF250_07_07_25, err_LPF250_07_07_25, _, meta_on = compute_oadev_from_res(res_LPF250_07_07_25, taus=taus_OFF_07_07_25)

In [395]:
res_LPF50_07_07_25 = resample_to_uniform(LPF50_07_07_25_ts, LPF50_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF50_07_07_25, err_LPF50_07_07_25, _, meta_on = compute_oadev_from_res(res_LPF50_07_07_25, taus=taus_OFF_07_07_25)

In [397]:
res_LPFNO_07_07_25 = resample_to_uniform(LPFNO_07_07_25_ts, LPFNO_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPFNO_07_07_25, err_LPFNO_07_07_25, _, meta_on = compute_oadev_from_res(res_LPFNO_07_07_25, taus=taus_OFF_07_07_25)

In [401]:
plt.figure()
plt.loglog(taus_OFF_07_07_25, adev_OFF_07_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_07_07_25, adev_LPF1000_07_07_25,  label="LPF1000")
plt.loglog(taus_OFF_07_07_25, adev_LPF750_07_07_25,  label="LPF750")
plt.loglog(taus_OFF_07_07_25, adev_LPF500_07_07_25,  label="LPF500")
plt.loglog(taus_OFF_07_07_25, adev_LPF250_07_07_25,  label="LPF250")
plt.loglog(taus_OFF_07_07_25, adev_LPF50_07_07_25,  label="LPF50")
plt.loglog(taus_OFF_07_07_25, adev_LPFNO_07_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [403]:
plt.figure()

scale=adev_OFF_07_07_25
plt.loglog(taus_OFF_07_07_25, scale/adev_OFF_07_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPF1000_07_07_25,  label="LPF1000")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPF750_07_07_25,  label="LPF750")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPF500_07_07_25,  label="LPF500")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPF250_07_07_25,  label="LPF250")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPF50_07_07_25,  label="LPF50")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPFNO_07_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [508]:
FSM_LPFcomparison_07_07_25=[(taus_OFF_07_07_25, adev_OFF_07_07_25),
                            (taus_OFF_07_07_25, adev_LPF1000_07_07_25),
                            (taus_OFF_07_07_25, adev_LPF750_07_07_25),
                            (taus_OFF_07_07_25, adev_LPF500_07_07_25),
                            (taus_OFF_07_07_25, adev_LPF250_07_07_25),
                            (taus_OFF_07_07_25, adev_LPF50_07_07_25),
                            (taus_OFF_07_07_25, adev_LPFNO_07_07_25)]

In [510]:
np.save('FSM_LPFcomparison_07_07_25',FSM_LPFcomparison_07_07_25)

In [405]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_NF450_Q15_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
NF450_Q15_07_07_25_ts=ts
NF450_Q15_07_07_25_xs=xs

In [410]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_NF300_Q15_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
NF300_Q15_07_07_25_ts=ts
NF300_Q15_07_07_25_xs=xs

In [412]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-07\PID_1_001_0_ON_NF150_Q15_07_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
NF150_Q15_07_07_25_ts=ts
NF150_Q15_07_07_25_xs=xs

In [407]:
res_NF450_Q15_07_07_25 = resample_to_uniform(NF450_Q15_07_07_25_ts, NF450_Q15_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_NF450_Q15_07_07_25, err_NF450_Q15_07_07_25, _, meta_on = compute_oadev_from_res(res_NF450_Q15_07_07_25, taus=taus_OFF_07_07_25)

In [414]:
res_NF300_Q15_07_07_25 = resample_to_uniform(NF300_Q15_07_07_25_ts, NF300_Q15_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_NF300_Q15_07_07_25, err_NF300_Q15_07_07_25, _, meta_on = compute_oadev_from_res(res_NF300_Q15_07_07_25, taus=taus_OFF_07_07_25)

In [416]:
res_NF150_Q15_07_07_25 = resample_to_uniform(NF150_Q15_07_07_25_ts, NF150_Q15_07_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_NF150_Q15_07_07_25, err_NF150_Q15_07_07_25, _, meta_on = compute_oadev_from_res(res_NF150_Q15_07_07_25, taus=taus_OFF_07_07_25)

In [418]:
plt.figure()
plt.loglog(taus_OFF_07_07_25, adev_OFF_07_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_07_07_25, adev_NF450_Q15_07_07_25,  label="NF450_Q15")
plt.loglog(taus_OFF_07_07_25, adev_NF300_Q15_07_07_25,  label="NF300_Q15")
plt.loglog(taus_OFF_07_07_25, adev_NF150_Q15_07_07_25,  label="NF150_Q15")
plt.loglog(taus_OFF_07_07_25, adev_LPFNO_07_07_25,  label="NFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [421]:
plt.figure()

scale=adev_OFF_07_07_25
plt.loglog(taus_OFF_07_07_25, scale/adev_OFF_07_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_07_07_25, scale/adev_NF450_Q15_07_07_25,  label="NF450_Q15")
plt.loglog(taus_OFF_07_07_25, scale/adev_NF300_Q15_07_07_25,  label="NF300_Q15")
plt.loglog(taus_OFF_07_07_25, scale/adev_NF150_Q15_07_07_25,  label="NF150_Q15")
plt.loglog(taus_OFF_07_07_25, scale/adev_LPFNO_07_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [504]:
FSM_NFcomparison_07_07_25=[(taus_OFF_07_07_25, adev_OFF_07_07_25),
                           (taus_OFF_07_07_25, adev_NF450_Q15_07_07_25),
                           (taus_OFF_07_07_25, adev_NF300_Q15_07_07_25),
                           (taus_OFF_07_07_25, adev_NF150_Q15_07_07_25),
                           (taus_OFF_07_07_25, adev_LPFNO_07_07_25)]

In [506]:
np.save('FSM_NFcomparison_07_07_25',FSM_NFcomparison_07_07_25)

In [423]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_0_0_0_OFF_LPFNO_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
OFF_15_07_25_ts=ts
OFF_15_07_25_xs=xs

In [425]:
taus_OFF_15_07_25, adev_OFF_15_07_25, err_OFF_15_07_25, _, meta_OFF_15_07_25 = compute_oadev_from_series(OFF_15_07_25_ts, OFF_15_07_25_xs, tolerance_ppm=500)

In [427]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0_ON_LPF1000_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF1000_15_07_25_ts=ts
LPF1000_15_07_25_xs=xs

In [429]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0_ON_LPF100_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF100_15_07_25_ts=ts
LPF100_15_07_25_xs=xs

In [431]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0_ON_LPF10_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF10_15_07_25_ts=ts
LPF10_15_07_25_xs=xs

In [433]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0_ON_LPF1_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPF1_15_07_25_ts=ts
LPF1_15_07_25_xs=xs

In [435]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0_ON_LPFNO_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
LPFNO_15_07_25_ts=ts
LPFNO_15_07_25_xs=xs

In [437]:
res_LPF1000_15_07_25 = resample_to_uniform(LPF1000_15_07_25_ts, LPF1000_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF1000_15_07_25, err_LPF1000_15_07_25, _, meta_on = compute_oadev_from_res(res_LPF1000_15_07_25, taus=taus_OFF_15_07_25)

In [439]:
res_LPF100_15_07_25 = resample_to_uniform(LPF100_15_07_25_ts, LPF100_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF100_15_07_25, err_LPF100_15_07_25, _, meta_on = compute_oadev_from_res(res_LPF100_15_07_25, taus=taus_OFF_15_07_25)

In [441]:
res_LPF10_15_07_25 = resample_to_uniform(LPF10_15_07_25_ts, LPF10_15_07_25_xs, tolerance_ppm=10, method="linear")
_, adev_LPF10_15_07_25, err_LPF10_15_07_25, _, meta_on = compute_oadev_from_res(res_LPF10_15_07_25, taus=taus_OFF_15_07_25)

In [443]:
res_LPF1_15_07_25 = resample_to_uniform(LPF1_15_07_25_ts, LPF1_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPF1_15_07_25, err_LPF1_15_07_25, _, meta_on = compute_oadev_from_res(res_LPF1_15_07_25, taus=taus_OFF_15_07_25)

In [445]:
res_LPFNO_15_07_25 = resample_to_uniform(LPFNO_15_07_25_ts, LPFNO_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_LPFNO_15_07_25, err_LPFNO_15_07_25, _, meta_on = compute_oadev_from_res(res_LPFNO_15_07_25, taus=taus_OFF_15_07_25)

In [448]:
plt.figure()
plt.loglog(taus_OFF_15_07_25, adev_OFF_15_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_15_07_25, adev_LPF1000_15_07_25,  label="LPF1000")
plt.loglog(taus_OFF_15_07_25, adev_LPF100_15_07_25,  label="LPF100")
plt.loglog(taus_OFF_15_07_25, adev_LPF10_15_07_25,  label="LPF10")
plt.loglog(taus_OFF_15_07_25, adev_LPF1_15_07_25,  label="LPF1")
plt.loglog(taus_OFF_15_07_25, adev_LPFNO_15_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [500]:
FSM_LPFcomparison_15_07_25=[(taus_OFF_15_07_25, adev_OFF_15_07_25),
                            (taus_OFF_15_07_25, adev_LPF1000_15_07_25),
                            (taus_OFF_15_07_25, adev_LPF100_15_07_25),
                            (taus_OFF_15_07_25, adev_LPF10_15_07_25),
                            (taus_OFF_15_07_25, adev_LPF1_15_07_25),
                            (taus_OFF_15_07_25, adev_LPFNO_15_07_25)]

In [502]:
np.save('FSM_LPFcomparison_15_07_25',FSM_LPFcomparison_15_07_25)

In [450]:
plt.figure()

scale=adev_OFF_15_07_25
plt.loglog(taus_OFF_15_07_25, scale/adev_OFF_15_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPF1000_15_07_25,  label="LPF1000")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPF100_15_07_25,  label="LPF100")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPF10_15_07_25,  label="LPF10")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPF1_15_07_25,  label="LPF1")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPFNO_15_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [452]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_01_ON_LPFNO_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
D_0100_15_07_25_ts=ts
D_0100_15_07_25_xs=xs
res_D_0100_15_07_25 = resample_to_uniform(D_0100_15_07_25_ts, D_0100_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_D_0100_15_07_25, err_D_0100_15_07_25, _, meta_on = compute_oadev_from_res(res_D_0100_15_07_25, taus=taus_OFF_15_07_25)

In [454]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_001_ON_LPFNO_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
D_0010_15_07_25_ts=ts
D_0010_15_07_25_xs=xs
res_D_0010_15_07_25 = resample_to_uniform(D_0010_15_07_25_ts, D_0010_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_D_0010_15_07_25, err_D_0010_15_07_25, _, meta_on = compute_oadev_from_res(res_D_0010_15_07_25, taus=taus_OFF_15_07_25)

In [456]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-15\PID_1_003_0001_ON_LPFNO_15_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
D_0001_15_07_25_ts=ts
D_0001_15_07_25_xs=xs
res_D_0001_15_07_25 = resample_to_uniform(D_0001_15_07_25_ts, D_0001_15_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_D_0001_15_07_25, err_D_0001_15_07_25, _, meta_on = compute_oadev_from_res(res_D_0001_15_07_25, taus=taus_OFF_15_07_25)

In [458]:
plt.figure()
plt.loglog(taus_OFF_15_07_25, adev_OFF_15_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_15_07_25, adev_LPFNO_15_07_25,  label="D_0000")
plt.loglog(taus_OFF_15_07_25, adev_D_0001_15_07_25,  label="D_0001")
plt.loglog(taus_OFF_15_07_25, adev_D_0010_15_07_25,  label="D_0010")
plt.loglog(taus_OFF_15_07_25, adev_D_0100_15_07_25,  label="D_0100")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [460]:
plt.figure()

scale=adev_OFF_15_07_25
plt.loglog(taus_OFF_15_07_25, scale/adev_OFF_15_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_15_07_25, scale/adev_LPFNO_15_07_25,  label="D_0000")
plt.loglog(taus_OFF_15_07_25, scale/adev_D_0001_15_07_25,  label="D_0001")
plt.loglog(taus_OFF_15_07_25, scale/adev_D_0010_15_07_25,  label="D_0010")
plt.loglog(taus_OFF_15_07_25, scale/adev_D_0100_15_07_25,  label="D_0100")


plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [463]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-16\PID_0_0_0_OFF_LPFNO_16_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
OFF_16_07_25_ts=ts
OFF_16_07_25_xs=xs

In [465]:
taus_OFF_16_07_25, adev_OFF_16_07_25, err_OFF_16_07_25, _, meta_OFF_16_07_25 = compute_oadev_from_series(OFF_16_07_25_ts, OFF_16_07_25_xs, tolerance_ppm=500)

In [467]:
ts, xs = np.loadtxt(r'D:\PhD Thesis Writing\Data\2025-07-16\PID_1_003_0_ON_LPFNO_16_07_25.csv',skiprows=23, delimiter=',',usecols =(0, 1), unpack = True, encoding='utf-8')
ts=ts*(10**(-3))
ON_16_07_25_ts=ts
ON_16_07_25_xs=xs

In [469]:
res_ON_16_07_25 = resample_to_uniform(ON_16_07_25_ts, ON_16_07_25_xs, tolerance_ppm=500, method="linear")
_, adev_ON_16_07_25, err_ON_16_07_25, _, meta_on = compute_oadev_from_res(res_ON_16_07_25, taus=taus_OFF_16_07_25)

In [481]:
plt.figure()
plt.loglog(taus_OFF_16_07_25, adev_OFF_16_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_16_07_25, adev_ON_16_07_25,  label="FSM ON, tuned")



plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [479]:
plt.figure()
plt.loglog(taus_OFF_15_07_25, adev_OFF_15_07_25,  label="FSM OFF")
plt.loglog(taus_OFF_15_07_25, adev_LPFNO_15_07_25,  label="LPFNO")

plt.xlabel("Averaging time τ [s]")
plt.ylabel("Overlapping Allan deviation σ(τ)")
plt.grid(True, which="both")
plt.legend()
plt.show()

In [483]:
plot_oadev_compare2(taus_OFF_16_07_25, adev_OFF_16_07_25, adev_ON_16_07_25, adeverr_off=err_OFF_16_07_25, adeverr_on=err_ON_16_07_25)

In [485]:
FSM_Final_16_07_25=[(taus_OFF_16_07_25, adev_OFF_16_07_25),
                    (taus_OFF_16_07_25, adev_ON_16_07_25)]

In [487]:
np.save('FSM_Final_16_07_25',FSM_Final_16_07_25)

In [496]:
FSM_Dcomparison_15_07_25=[(taus_OFF_15_07_25, adev_OFF_15_07_25),
                          (taus_OFF_15_07_25, adev_LPFNO_15_07_25),
                          (taus_OFF_15_07_25, adev_D_0001_15_07_25),
                          (taus_OFF_15_07_25, adev_D_0010_15_07_25),
                          (taus_OFF_15_07_25, adev_D_0100_15_07_25)]

In [498]:
np.save('FSM_Dcomparison_15_07_25',FSM_Dcomparison_15_07_25)